In [ ]:
import pandas as pd
import requests
import torch
import io

In [ ]:
url = "https://raw.githubusercontent.com/neuraloperator/neuraloperator/main/neuralop/data/datasets/data/burgers_train_16.pt"
# neuralop/data/datasets/data/burgers_test_16.pt
# https://github.com/neuraloperator/neuraloperator/blob/98cd305099f4a2b232ed85773984f3e5991f9b1a/neuralop/data/datasets/data/burgers_test_16.pt

In [ ]:
# Getting the content from the web
response = requests.get(url)
response.raise_for_status() # Checking for errors
print(response.content[:100]) # Print the first 100 characters

b'PK\x03\x04\x00\x00\x08\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x19\x00\t\x00burgers_16_train/data.pklFB\x05\x00ZZZZZ\x80\x02}q\x00(X\x01\x00\x00\x00xq\x01ctorch._utils\n_rebuild'


In [ ]:
buffer = io.BytesIO(response.content)

In [50]:
data = torch.load(buffer, map_location = 'cpu', weights_only = False)

In [51]:
if isinstance(data, dict):
    print("Dataset keys:", data.keys())
    # Typically you'll see something like: dict_keys(['x', 'y', 't'])
else:
    print("Data type:", type(data))

Dataset keys: dict_keys(['x', 'y', 'visc'])


In [52]:
x_data = data['x']
print(x_data)

tensor([[ 0.1509,  0.0968,  0.0171,  ...,  0.1103,  0.1604,  0.1723],
        [-0.0673,  0.2881,  0.5526,  ..., -0.8302, -0.7130, -0.4337],
        [-0.0720, -0.0110,  0.0552,  ..., -0.2151, -0.1944, -0.1389],
        ...,
        [ 0.1214,  0.1569,  0.1438,  ..., -0.0085,  0.0062,  0.0565],
        [ 0.0134,  0.0186,  0.0165,  ..., -0.0695, -0.0365, -0.0082],
        [-0.7555, -0.8089, -0.7338,  ..., -0.0290, -0.3280, -0.5833]],
       dtype=torch.float64)


In [53]:
import jax.numpy as jnp

In [54]:
x_np = x_data.detach().cpu().numpy()

In [55]:
x_input = jnp.asarray(x_np)

In [56]:
print(x_input)

[[ 0.15092067  0.0968335   0.0171366  ...  0.11032616  0.16038279
   0.17227554]
 [-0.06729478  0.28813443  0.55255806 ... -0.8301725  -0.7130197
  -0.43373397]
 [-0.07195096 -0.01100656  0.05521011 ... -0.21508782 -0.1943769
  -0.13892312]
 ...
 [ 0.12142732  0.15689726  0.1437555  ... -0.00850507  0.00620764
   0.05650266]
 [ 0.01343963  0.01862456  0.01652554 ... -0.06954313 -0.03651619
  -0.00824137]
 [-0.7554656  -0.80886483 -0.73384076 ... -0.02903297 -0.3279872
  -0.58334357]]


In [58]:
print(data_jax)

{'x': Array([[ 0.15092067,  0.0968335 ,  0.0171366 , ...,  0.11032616,
         0.16038279,  0.17227554],
       [-0.06729478,  0.28813443,  0.55255806, ..., -0.8301725 ,
        -0.7130197 , -0.43373397],
       [-0.07195096, -0.01100656,  0.05521011, ..., -0.21508782,
        -0.1943769 , -0.13892312],
       ...,
       [ 0.12142732,  0.15689726,  0.1437555 , ..., -0.00850507,
         0.00620764,  0.05650266],
       [ 0.01343963,  0.01862456,  0.01652554, ..., -0.06954313,
        -0.03651619, -0.00824137],
       [-0.7554656 , -0.80886483, -0.73384076, ..., -0.02903297,
        -0.3279872 , -0.58334357]], dtype=float32), 'y': Array([[[ 1.50920674e-01,  9.68334973e-02,  1.71365980e-02, ...,
          1.10326163e-01,  1.60382792e-01,  1.72275543e-01],
        [ 1.50650725e-01,  9.88628715e-02,  1.80344768e-02, ...,
          1.02428228e-01,  1.49633780e-01,  1.67745203e-01],
        [ 1.49742037e-01,  1.00363642e-01,  1.84560157e-02, ...,
          9.57828760e-02,  1.40575692e-01, 

In [40]:
import os
import orbax.checkpoint as ocp

# 1. Convert the relative name to an absolute path
checkpoint_path = os.path.abspath('my_fno_checkpoint')

# 2. Initialize the checkpointer
checkpointer = ocp.StandardCheckpointer()

# 3. Save using the absolute path
# Make sure data_jax is a dictionary of JAX arrays
checkpointer.save(checkpoint_path, data_jax)

print(f"Dataset successfully saved to: {checkpoint_path}")

Dataset successfully saved to: /home/ziq03/learning/sharad/my_fno_checkpoint


In [41]:
restored_data = checkpointer.restore(checkpoint_path)

In [42]:
x_train = restored_data['x']

In [44]:
type(restored_data)

dict

In [46]:
loaded_data = np.load('burgers_test_16.npz')

In [59]:
np.savez_compressed('burgers_train_16.npz', **data_jax)

In [1]:
import scipy.io
import jax.numpy as jnp
data = scipy.io.loadmat('burgers_data_R10.mat')
print("Dataset keys:", data.keys())

# Convert every tensor in the 'data' dictionary to a JAX array
data_jax = {k:v for k,v in data.items()}
print(data_jax.keys())

FileNotFoundError: [Errno 2] No such file or directory: 'burgers_data_R10.mat'

In [8]:
print(len(data_jax['a_x']))

2048


In [3]:
data_jax = {k:v for k,v in data.items()}

In [13]:
print(data_jax.keys())

dict_keys(['__header__', '__version__', '__globals__', 'a', 'a_smooth', 'a_smooth_x', 'a_x', 'u'])


In [16]:
import scipy.io
import jax.numpy as jnp
from pathlib import Path

# Get the directory where this config file is located
# config_dir = Path(__file__).parent
# # Navigate to data folder in parent directory
# data_path = config_dir.parent / 'data' / 'burgers_v100_t100_r1024_N2048.mat'
# data = scipy.io.loadmat(data_path)

data = scipy.io.loadmat('/home/zenteiq/Documents/SciREX/data/burgers_v100_t100_r1024_N2048.mat')

/home/zenteiq/Documents/SciREX/.venv/lib/python3.12/site-packages/scipy/io/matlab/_mio.py:236: MatReadWarning: Duplicate variable name "None" in stream - replacing previous with new
Considerscipy.io.matlab.varmats_from_mat to split file into single variable files
  matfile_dict = MR.get_variables(variable_names)


In [17]:
# print(len(data['steps']))
for key in data.keys():
    print(f"{key}: {len(data[key])}")

__header__: 76
__version__: 3
__globals__: 0
input: 2048
output: 2048
sigma: 1
steps: 1
tau: 1
tspan: 1
None: 1
u0eval: 1
visc: 1
__function_workspace__: 1


In [18]:
print(data['output'].shape)
print(data['input'].shape)
print(data['tspan'].shape)
print(data['tspan'])
print(data['tau'])
print(data['sigma'])
print(data['u0eval'])
print(data['steps'])
print(data['None'])
print(data['input'][0])
print(data['output'][0])

(2048, 101, 1024)
(2048, 1024)
(1, 101)
[[0.   0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1  0.11 0.12 0.13
  0.14 0.15 0.16 0.17 0.18 0.19 0.2  0.21 0.22 0.23 0.24 0.25 0.26 0.27
  0.28 0.29 0.3  0.31 0.32 0.33 0.34 0.35 0.36 0.37 0.38 0.39 0.4  0.41
  0.42 0.43 0.44 0.45 0.46 0.47 0.48 0.49 0.5  0.51 0.52 0.53 0.54 0.55
  0.56 0.57 0.58 0.59 0.6  0.61 0.62 0.63 0.64 0.65 0.66 0.67 0.68 0.69
  0.7  0.71 0.72 0.73 0.74 0.75 0.76 0.77 0.78 0.79 0.8  0.81 0.82 0.83
  0.84 0.85 0.86 0.87 0.88 0.89 0.9  0.91 0.92 0.93 0.94 0.95 0.96 0.97
  0.98 0.99 1.  ]]
[[7]]
[[49]]
[[0.28710181 0.2884588  0.28989018 ... 0.28448161 0.28578716 0.28710181]]
[[100]]
[(b'u0', b'MCOS', b'chebfun', array([[3707764736],
        [         2],
        [         1],
        [         1],
        [       709],
        [         5]], dtype=uint32))             ]
[-0.62691281 -0.62563444 -0.62439611 ... -0.63105143 -0.629645
 -0.62825371]
[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 

In [26]:
print(len(data['tspan']))
print(data['tspan'].shape)
print(data['tspan'])

1
(1, 101)
[[0.   0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1  0.11 0.12 0.13
  0.14 0.15 0.16 0.17 0.18 0.19 0.2  0.21 0.22 0.23 0.24 0.25 0.26 0.27
  0.28 0.29 0.3  0.31 0.32 0.33 0.34 0.35 0.36 0.37 0.38 0.39 0.4  0.41
  0.42 0.43 0.44 0.45 0.46 0.47 0.48 0.49 0.5  0.51 0.52 0.53 0.54 0.55
  0.56 0.57 0.58 0.59 0.6  0.61 0.62 0.63 0.64 0.65 0.66 0.67 0.68 0.69
  0.7  0.71 0.72 0.73 0.74 0.75 0.76 0.77 0.78 0.79 0.8  0.81 0.82 0.83
  0.84 0.85 0.86 0.87 0.88 0.89 0.9  0.91 0.92 0.93 0.94 0.95 0.96 0.97
  0.98 0.99 1.  ]]


In [27]:
length = data['tspan'].shape[1]
print(length)


101


In [13]:
import gc

del data
gc.collect()
print(data.keys())

NameError: name 'data' is not defined

In [28]:
input_function = data['input']
print(input_function)
input_function = input_function.reshape(2048,1024,1)
print(input_function.shape)

[[-0.62691281 -0.62563444 -0.62439611 ... -0.63105143 -0.629645
  -0.62825371]
 [-0.28598222 -0.28767795 -0.28943569 ... -0.2813008  -0.28281842
  -0.28436591]
 [ 0.22911716  0.23143463  0.23378511 ...  0.22252161  0.22469419
   0.22688881]
 ...
 [-0.08235099 -0.07872406 -0.07501566 ... -0.09259651 -0.0892875
  -0.08586652]
 [-0.25676794 -0.25294933 -0.24917613 ... -0.26803583 -0.26431217
  -0.26057866]
 [ 0.28710181  0.2884588   0.28989018 ...  0.28320862  0.28448161
   0.28578716]]
(2048, 1024, 1)


In [7]:
import jax.numpy as jnp
input_function = jnp.zeros((2048,1024))
input_function = input_function.reshape(2048,1024,1)
print(input_function.shape)

(2048, 1024, 1)


In [ ]:
x_grid = jnp.linspace(0, 2*jnp.pi, 1024)
print(x_grid)
print(len(x_grid))
input_channels = jnp.append(x_grid, )

[0.0000000e+00 6.1419215e-03 1.2283843e-02 ... 6.2709017e+00 6.2770438e+00
 6.2831855e+00]
1024


In [29]:
jnp.tile(x_grid, (2048,1))

Array([[0.0000000e+00, 6.1419215e-03, 1.2283843e-02, ..., 6.2709017e+00,
        6.2770438e+00, 6.2831855e+00],
       [0.0000000e+00, 6.1419215e-03, 1.2283843e-02, ..., 6.2709017e+00,
        6.2770438e+00, 6.2831855e+00],
       [0.0000000e+00, 6.1419215e-03, 1.2283843e-02, ..., 6.2709017e+00,
        6.2770438e+00, 6.2831855e+00],
       ...,
       [0.0000000e+00, 6.1419215e-03, 1.2283843e-02, ..., 6.2709017e+00,
        6.2770438e+00, 6.2831855e+00],
       [0.0000000e+00, 6.1419215e-03, 1.2283843e-02, ..., 6.2709017e+00,
        6.2770438e+00, 6.2831855e+00],
       [0.0000000e+00, 6.1419215e-03, 1.2283843e-02, ..., 6.2709017e+00,
        6.2770438e+00, 6.2831855e+00]], dtype=float32)

In [32]:
x_single = jnp.linspace(0,2*jnp.pi, 1024)
x_grid = jnp.tile(x_single, (2048,1))
u0_input = jnp.array(data['input'])
input_data = jnp.stack([x_grid, u0_input], axis=2)

In [33]:
print(input_data.shape)

(2048, 1024, 2)


In [ ]:
import requests
import h5py
import io

# Replace with your actual Zenodo file URL
zenodo_url = "https://zenodo.org/records/YOUR_RECORD_ID/files/Burgers_4096.h5"

# Download file to memory
response = requests.get(zenodo_url)
response.raise_for_status()  # Raise error if download failed

# Load into memory as a file-like object
file_bytes = io.BytesIO(response.content)

# Open and read with h5py
with h5py.File(file_bytes, 'r') as f:
    # List all keys in the file
    print(f.keys())
    
    # Access data (example - adjust key names as needed)
    data = f['your_dataset_key'][:]  # [:] loads into memory

Reading data from 1D/Burgers_4096.h5...


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '1D/Burgers_4096.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)